In [35]:
import pandas as pd
import numpy as np

student_assessment = pd.read_csv("studentAssessment.csv")
assessments = pd.read_csv("assessments.csv")


print(f"studentAssessment.csv: {len(student_assessment)} rows")
print(f"assessments.csv: {len(assessments)} rows")
student_assessment.head()

studentAssessment.csv: 173912 rows
assessments.csv: 206 rows


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78
1,1752,28400,22,0,70
2,1752,31604,17,0,72
3,1752,32885,26,0,69
4,1752,38053,19,0,79


In [36]:
student_assessment["score"] = pd.to_numeric(student_assessment["score"], errors="coerce")
student_assessment["date_submitted"] = pd.to_numeric(student_assessment["date_submitted"], errors="coerce")
assessments["date"] = pd.to_numeric(assessments["date"], errors="coerce")

merged = student_assessment.merge(assessments, on="id_assessment", how="left")
merged = merged.dropna(subset=["score"]).sort_values(["id_student", "date_submitted"])

def bucket(score):
    if score >= 80: return "Hard"
    elif score >= 50: return "Medium"
    else: return "Easy"

rows = []
for student_id, group in merged.groupby("id_student"):
    group = group.reset_index(drop=True)
    prev_score, attempts, prev_difficulty = None, 0, None
    for _, r in group.iterrows():
        attempts += 1
        score = float(r["score"])
        due, submitted = r.get("date"), r.get("date_submitted")
        try:
            time_proxy = abs(float(submitted) - float(due)) if pd.notna(due) and pd.notna(submitted) else 30.0
        except (TypeError, ValueError):
            time_proxy = 30.0
        avg_time = max(5.0, min(120.0, time_proxy * 2))
        if prev_score is not None:
            rows.append({
                "previous_score": prev_score, "attempts": attempts,
                "avg_time_seconds": avg_time, "previous_difficulty": prev_difficulty,
                "next_difficulty": bucket(score),
            })
        prev_score, prev_difficulty = score, bucket(score)

df = pd.DataFrame(rows)
print(f"Total training rows: {len(df)}")
df.head()

Total training rows: 150388


,previous_score,attempts,avg_time_seconds,previous_difficulty,next_difficulty
0,60.0,2,6.0,Medium,Easy
1,48.0,3,5.0,Easy,Medium
2,63.0,4,5.0,Medium,Medium
3,61.0,5,10.0,Medium,Medium
4,93.0,2,84.0,Hard,Hard


In [37]:
df["next_difficulty"].value_counts()

,count
next_difficulty,
Hard,77425
Medium,60019
Easy,12944


In [38]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

NUMERIC = ["previous_score", "attempts", "avg_time_seconds"]
CATEGORICAL = ["previous_difficulty"]

df["previous_difficulty"] = df["previous_difficulty"].fillna("Easy")
X = df[NUMERIC + CATEGORICAL]
y = df["next_difficulty"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
])

models = {
    "LogisticRegression": Pipeline([("prep", preprocessor), ("model", LogisticRegression(max_iter=1000))]),
    "DecisionTreeClassifier": Pipeline([("prep", preprocessor), ("model", DecisionTreeClassifier(max_depth=8, random_state=42))]),
    "RandomForestClassifier": Pipeline([("prep", preprocessor), ("model", RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42))]),
    "GradientBoosting": Pipeline([("prep", preprocessor), ("model", GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42))]),
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, average="macro", zero_division=0),
        "recall": recall_score(y_test, preds, average="macro", zero_division=0),
        "f1": f1_score(y_test, preds, average="macro", zero_division=0),
    }
    results[name] = metrics
    print(f"\n--- {name} ---")
    for k, v in metrics.items():
        print(f"{k:>10}: {v:.4f}")
    print(classification_report(y_test, preds, zero_division=0))

best_name = max(results, key=lambda n: results[n]["accuracy"])
print(f"\n{'='*40}")
print(f"BEST MODEL: {best_name} (Accuracy = {results[best_name]['accuracy']:.4f})")
print(f"{'='*40}")


--- LogisticRegression ---
  accuracy: 0.5891
 precision: 0.5110
    recall: 0.4391
        f1: 0.4393
              precision    recall  f1-score   support

        Easy       0.37      0.07      0.12      2589
        Hard       0.65      0.73      0.69     15485
      Medium       0.51      0.52      0.51     12004

    accuracy                           0.59     30078
   macro avg       0.51      0.44      0.44     30078
weighted avg       0.57      0.59      0.57     30078


--- DecisionTreeClassifier ---
  accuracy: 0.6199
 precision: 0.5718
    recall: 0.4924
        f1: 0.5076
              precision    recall  f1-score   support

        Easy       0.49      0.18      0.26      2589
        Hard       0.68      0.75      0.71     15485
      Medium       0.55      0.55      0.55     12004

    accuracy                           0.62     30078
   macro avg       0.57      0.49      0.51     30078
weighted avg       0.61      0.62      0.61     30078


--- RandomForestClassifie

In [39]:
comparison = pd.DataFrame(results).T
comparison

,accuracy,precision,recall,f1
LogisticRegression,0.589068,0.510996,0.439145,0.439252
DecisionTreeClassifier,0.619855,0.571781,0.492379,0.507552
RandomForestClassifier,0.619090,0.570357,0.479218,0.490953
GradientBoosting,0.626438,0.578204,0.487460,0.500170


In [40]:
import joblib
import os

# Create folder if it doesn't exist
os.makedirs("app/ml", exist_ok=True)

joblib.dump({
    "pipeline": models[best_name],
    "model_name": best_name,
    "metrics": results[best_name],
    "all_results": results,
    "features": NUMERIC + CATEGORICAL,
    "classes": sorted(y.unique().tolist()),
}, "app/ml/difficulty_model.joblib")

print(" Saved to app/ml/difficulty_model.joblib")

 Saved to app/ml/difficulty_model.joblib


In [41]:
import os
print("Current directory:", os.getcwd())
print("\nFiles in current directory:")
print(os.listdir('.'))

Current directory: /content

Files in current directory:
['.config', 'app', 'assessments.csv', 'studentAssessment.csv', 'sample_data']


In [43]:
from google.colab import files
files.download("app/ml/difficulty_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>